# Download and setup data

This notebook walks through the process of loading and splitting the datasets used in this paper.

Datasets:
- News-Commentary
- EMEA
- OpenSubtitiles
en-fr datasets via HuggingFace

Configs for e.g. the eval split etc are provided in the corresponding config/{dataset}_enfr.yaml file



In [1]:
import os
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import load_dataset
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 

device = torch.device("cpu")

In [3]:

ds = load_dataset("Helsinki-NLP/news_commentary", "en-fr")

ds

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 209479
    })
})

In [4]:
print(ds)
print("Splits:", list(ds.keys()))
print("Columns:", ds[list(ds.keys())[0]].column_names)

ex = ds[list(ds.keys())[0]][0]
ex


DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 209479
    })
})
Splits: ['train']
Columns: ['id', 'translation']


{'id': '0',
 'translation': {'en': '$10,000 Gold?',
  'fr': 'L’or à 10.000 dollars l’once\xa0?'}}

In [5]:
EVAL_SIZE = 300
SPLIT_NAME = list(ds.keys())[0]

data = ds[SPLIT_NAME]

data_shuf = data.shuffle(seed=SEED)
eval_ds = data_shuf.select(range(min(EVAL_SIZE, len(data_shuf))))

rows = []
for i in range(len(eval_ds)):
    tr = eval_ds[i]["translation"]
    rows.append({
        "id": i,
        "src_en": tr["en"],
        "ref_fr": tr["fr"],
    })

eval_df = pd.DataFrame(rows)
eval_df.head(), len(eval_df)


(   id                                             src_en  \
 0   0  It may be that the SNB merely acknowledges wha...   
 1   1  An alternative to this line of argument is to ...   
 2   2  Simultaneous progress on both substance and pr...   
 3   3  These figures were calculated by economists Ch...   
 4   4  Most Asian countries are recovering strongly f...   
 
                                               ref_fr  
 0  Il est possible que la BNS reconnaisse ce que ...  
 1  Dans cette même ligne de pensée, d’autres décr...  
 2  Des progrès simultanés, tant aux plans de la s...  
 3  Les économistes Chris Green et Isabel Galiana ...  
 4  La plupart des pays asiatiques se remettent tr...  ,
 300)

In [6]:
os.makedirs("data", exist_ok=True)
eval_path = "eval_enfr_newscommentary_seed42_n300.csv"
eval_df.to_csv(eval_path, index=False)
eval_path


'eval_enfr_newscommentary_seed42_n300.csv'

In [7]:
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model.eval()

SRC_LANG = "en_XX"
TGT_LANG = "fr_XX"

tokenizer.src_lang = SRC_LANG
forced_bos_token_id = tokenizer.lang_code_to_id[TGT_LANG] 

forced_bos_token_id


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

250008

In [ ]:
K = 8
MAX_NEW_TOKENS = 128

TOP_P = 0.9
TEMPERATURE = 1.0

def sample_one(src_text: str) -> str:
    inputs = tokenizer(src_text, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            do_sample=True,
            top_p=TOP_P,
            temperature=TEMPERATURE,
            max_new_tokens=MAX_NEW_TOKENS,
            forced_bos_token_id=forced_bos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Generate candidates (slow but OK for n=300, K=8 on CPU)
candidates = []
for row in tqdm(eval_df.itertuples(index=False), total=len(eval_df)):
    for k in range(K):
        hyp = sample_one(row.src_en)
        candidates.append({
            "id": int(row.id),
            "cand_id": k,
            "src_en": row.src_en,
            "ref_fr": row.ref_fr,
            "hyp_fr": hyp,
        })

cand_df = pd.DataFrame(candidates)
cand_df.head(), len(cand_df)


  0%|          | 0/300 [00:00<?, ?it/s]

In [ ]:
cand_path = "data/candidates_enfr_newscommentary_seed42_n300_k8.jsonl"
cand_df.to_json(cand_path, orient="records", lines=True, force_ascii=False)
cand_path


In [ ]:
# Show a few examples to ensure language direction is right
sample = cand_df[cand_df["id"].isin([0, 1])].copy()
sample[["id", "cand_id", "src_en", "hyp_fr"]].head(6)
